<a href="https://colab.research.google.com/github/ArinaKrasotina/data-analysis/blob/main/%D0%9A%D1%80%D0%B0%D1%81%D0%BE%D1%82%D0%B8%D0%BD%D0%B0_%22%D0%A2%D1%80%D0%B5%D1%82%D0%B8%D0%B9_%D0%BA%D0%B5%D0%B9%D1%81_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 3 кейс

**В этом кейсе вы будете рассчитывать:**
* retention
* rolling retention
* lifetime
* churn rate
* mau
* wau
* dau

**Важно**

Перед началом решения задачи выполните следующую ячейку - в ней скачиваются нужные файлы

In [ ]:
!wget https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv

!wget https://gist.github.com/Vs8th/aacb80595d1d6aaa2e31eb735f8bc644/raw/entries.csv

!wget https://gist.github.com/Vs8th/0e827e9a608117345dd6585ab81e8c86/raw/metrics.txt

--2025-11-19 09:36:45--  https://gist.github.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.github.com (gist.github.com)... 140.82.112.3
Connecting to gist.github.com (gist.github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv [following]
--2025-11-19 09:36:45--  https://gist.githubusercontent.com/Vs8th/739269a03f2f4a7396d04d6739da3771/raw/registrations.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14918 (15K) [text/plain]
Saving to: ‘registrations.csv’

registrations.csv   100%[===================>]  14.57K  --.-KB/s    in 0s      

2025-11-19 09:36:45 (94.3 M

Файлами для работы являются `registrations.csv` и `entries.csv`. В них хранятся данные о регистрациях пользователей и входа на платформу соответственно.

### **Посчитайте Retention 15 дня (в процентах) для пользователей, зарегистрированных в январе**

Cохраните результат в переменную `retention_15_day`

**Примечание:** результат округлите до 5 знаков после запятой

In [ ]:
import csv
from datetime import datetime, timedelta
from collections import defaultdict

def load_data(filename, delimiter=';'):
    data = []
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter=delimiter)
        for row in reader:
            data.append(dict(row))
    return data

registrations = load_data('registrations.csv', delimiter=';')
entries = load_data('entries.csv', delimiter=';')

Колонки в registrations: ['user_id', 'registration_date']
Колонки в entries: ['user_id', 'entry_date']
Первая строка registrations: {'user_id': '1', 'registration_date': '2021-01-01'}
Первая строка entries: {'user_id': '1', 'entry_date': '2021-01-01'}


In [ ]:
# Преобразуем даты
for row in registrations:
    row['date'] = datetime.strptime(row['registration_date'], '%Y-%m-%d')
    row['user_id'] = row['user_id']

for row in entries:
    row['date'] = datetime.strptime(row['entry_date'], '%Y-%m-%d')
    row['user_id'] = row['user_id']

print(f"Загружено {len(registrations)} регистраций и {len(entries)} записей о входах")

# Создаем словарь регистраций
user_registration = {}
for row in registrations:
    user_registration[row['user_id']] = row['date']

Загружено 1000 регистраций и 20734 записей о входах
Уникальных пользователей: 1000


In [ ]:
# 1. Retention 15 дня для январских пользователей
january_users = [user_id for user_id, reg_date in user_registration.items()
                if reg_date.month == 1]

users_registered = len(january_users)
print(f"Январских пользователей: {users_registered}")

# Находим пользователей, активных на 15 день
users_active_day15 = set()
for entry in entries:
    user_id = entry['user_id']
    if user_id in january_users:
        reg_date = user_registration[user_id]
        days_diff = (entry['date'] - reg_date).days
        if days_diff == 15:
            users_active_day15.add(user_id)

retention_15_day = round(len(users_active_day15) / users_registered * 100, 5) if users_registered > 0 else 0

print(f"Пользователей активных на 15 день: {len(users_active_day15)}")
print(f"Retention 15 дня: {retention_15_day}%")

Январских пользователей: 86
Пользователей активных на 15 день: 47
Retention 15 дня: 54.65116%


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
# Открываем файл с правильными ответами
with open('metrics.txt', 'r') as f:
    answers = f.read().split('\n')

correct_answer = float(answers[0])

try:
    assert retention_15_day == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Rolling-retention 30 дня (в процентах) для пользователей из той же когорты**

Сохраните результат в переменную `rolling_retention`

**Примечание:** результат округлите до 5 знаков после запятой

In [ ]:
# 2. Rolling-retention 30 дня
users_active_day30_plus = set()
for entry in entries:
    user_id = entry['user_id']
    if user_id in january_users:
        reg_date = user_registration[user_id]
        days_diff = (entry['date'] - reg_date).days
        if days_diff >= 30:
            users_active_day30_plus.add(user_id)

rolling_retention = round(len(users_active_day30_plus) / users_registered * 100, 5) if users_registered > 0 else 0
print(f"Rolling retention 30 дня: {rolling_retention}%")

Rolling retention 30 дня: 29.06977%


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[1])

try:
    assert rolling_retention == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Lifetime по всем пользователям, посчитанный как интеграл от n-day retention**

Сохраните результат в переменную `lifetime`

**Примечание:** результат округлите до 5 знаков после запятой

In [ ]:
# 3. Lifetime по ВСЕМ пользователям
all_users = list(user_registration.keys())
total_users = len(all_users)
print(f"Всего пользователей: {total_users}")

# Собираем активность всех пользователей
user_activity_days_all = defaultdict(set)
max_days_all = 0

for entry in entries:
    user_id = entry['user_id']
    if user_id in user_registration:
        reg_date = user_registration[user_id]
        days_diff = (entry['date'] - reg_date).days
        if days_diff >= 0:
            user_activity_days_all[user_id].add(days_diff)
            max_days_all = max(max_days_all, days_diff)

print(f"Максимальное количество дней отслеживания: {max_days_all}")

# Считаем n-day retention по всем пользователям
retention_by_day_all = []
for day in range(max_days_all + 1):
    active_count = 0
    for user_id in all_users:
        if day in user_activity_days_all[user_id]:
            active_count += 1
    retention_rate = active_count / total_users
    retention_by_day_all.append(retention_rate)

# Lifetime как интеграл (сумма) retention rates
lifetime = round(sum(retention_by_day_all), 5)
print(f"Lifetime: {lifetime}")

Всего пользователей: 1000
Максимальное количество дней отслеживания: 30
Lifetime: 14.804


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[2])

try:
    assert lifetime == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Churn rate 29 дня (в долях), посчитанный по всем пользователям**

Сохраните результат в переменную `churn_29`.  
Если будете считать CR от Retention, ведите расчет от Rolling Retention, таким образом, мы получим всех, кто не заходил в 29 день и после. Ведя расчет от обычного Retention, наоборот - получим только CR в 29 день.


In [ ]:
# 4. Churn rate 29 дня по ВСЕМ пользователям
users_active_day29_plus_all = set()
for entry in entries:
    user_id = entry['user_id']
    if user_id in user_registration:
        reg_date = user_registration[user_id]
        days_diff = (entry['date'] - reg_date).days
        if days_diff >= 29:
            users_active_day29_plus_all.add(user_id)

rolling_retention_29_all = len(users_active_day29_plus_all) / total_users
churn_29 = round(1 - rolling_retention_29_all, 5)
print(f"Churn rate 29 дня: {churn_29}")

Churn rate 29 дня: 0.509


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[3])

try:
    assert churn_29 == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau за последний месяц/неделю/день записей**

Сохраните результат в переменные `dec_mau`, `dec_wau`, `dec_dau` соответственно

**Примечание:** последний месяц записей - декабрь. Поэтому `mau` рассчитываем для декабря (2021 года), для `wau` берем последнюю неделю - с 25 по 31 декабря, и для `dau` соответственно последний день - 31 декабря.

In [ ]:
# 5. MAU за декабрь 2021
dec_mau_users = set()
for entry in entries:
    if entry['date'].month == 12 and entry['date'].year == 2021:
        dec_mau_users.add(entry['user_id'])
dec_mau = len(dec_mau_users)
print(f"December MAU: {dec_mau}")

December MAU: 133


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[4])

try:
    assert dec_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [ ]:
# WAU за последнюю неделю декабря (25-31)
wau_start = datetime(2021, 12, 25)
wau_end = datetime(2021, 12, 31)
dec_wau_users = set()
for entry in entries:
    if wau_start <= entry['date'] <= wau_end:
        dec_wau_users.add(entry['user_id'])
dec_wau = len(dec_wau_users)
print(f"December WAU: {dec_wau}")

December WAU: 84


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[5])

try:
    assert dec_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [ ]:
# DAU за 31 декабря
last_day = datetime(2021, 12, 31)
dec_dau_users = set()
for entry in entries:
    if entry['date'].date() == last_day.date():
        dec_dau_users.add(entry['user_id'])
dec_dau = len(dec_dau_users)
print(f"December DAU: {dec_dau}")

December DAU: 47


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[6])

try:
    assert dec_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


### **Посчитайте Mau, Wau, Dau усредненные**

Сохраните результат в переменные `avg_mau`, `avg_wau`, `avg_dau` соответственно

**Примечание:** результаты округлите до 5 знаков после запятой

In [ ]:
# 6. Усредненный MAU
monthly_active = defaultdict(set)
for entry in entries:
    month_key = (entry['date'].year, entry['date'].month)
    monthly_active[month_key].add(entry['user_id'])

avg_mau = round(sum(len(users) for users in monthly_active.values()) / len(monthly_active), 5)
print(f"Average MAU: {avg_mau}")

In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[7])

try:
    assert avg_mau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [ ]:
# Усредненный WAU
weekly_active = defaultdict(set)
for entry in entries:
    week_key = (entry['date'].year, entry['date'].isocalendar()[1])
    weekly_active[week_key].add(entry['user_id'])

avg_wau = round(sum(len(users) for users in weekly_active.values()) / len(weekly_active), 5)
print(f"Average WAU: {avg_wau}")

Average WAU: 89.86792


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[8])

try:
    assert avg_wau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


In [ ]:
# Усредненный DAU
daily_active = defaultdict(set)
for entry in entries:
    day_key = entry['date'].date()
    daily_active[day_key].add(entry['user_id'])

avg_dau = round(sum(len(users) for users in daily_active.values()) / len(daily_active), 5)
print(f"Average DAU: {avg_dau}")

Average DAU: 40.5589


In [ ]:
#@title ✏️ Проверка: чтобы проверить свое решение запустите код в этой ячейке
correct_answer = float(answers[9])

try:
    assert avg_dau == correct_answer
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
